# Pesquisa LoL | Análise lexical e visualizações

Este caderno usa Python para contar correspondências literais em 43 transcrições do canal Yiok. Não usa um modelo generativo para classificar as falas. As entradas vêm de uma versão fixa do [repositório](https://github.com/lidibelas/pesquisa_lol/tree/c4412033401006b7e94b73cde3a05c61e045c12b).

**No Colab:** abra o arquivo e vá a **Ambiente de execução → Executar tudo**. Não precisa configurar chave nem GPU. Por padrão, o caderno só baixa os CSVs públicos de entrada. Não grava no GitHub, não acessa Drive e não muda os arquivos de origem. O CSV e o gráfico aparecem na pasta `resultados_mol/` do ambiente de execução.

- **Sem frases de cena:** medida de referência. **Com frases de cena:** comparação com a mesma base.
- **Ocorrências** contam cada correspondência aceita. **Flags** contam segmentos com uma ou mais ocorrências principais (para uma descrição, 0 ou 1). Palavras excluídas e flags auxiliares não entram na taxa principal. Não se encaixa uma expressão de vários termos atravessando segmentos; variações não listadas e correções sintáticas não são inferidas.
- **Taxa por mil** = 1.000 × ocorrências principais ÷ total de palavras na fonte. A transcrição e a descrição do YouTube têm colunas e denominadores próprios. Descrição vazia fica sem medidas. URLs são excluídas somente das descrições.
- Os dados de visualizações são os valores registrados nos metadados originais. Spearman é calculado a partir das taxas sem arredondar, usando a média das posições quando há empate. Desfecho descritivo: uma correspondência não confirma misoginia, e associação não implica causa.

**Transparência:** ChatGPT auxiliou na preparação e no teste do código. O processamento e as estatísticas são calculados pelas funções abertas abaixo. Adapte o anexo de uso de IA aos usos efetivos no trabalho, como exige a disciplina.


In [ ]:
import csv
import hashlib
import io
import math
import os
import re
import statistics
import unicodedata
from collections import Counter
from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen

COMMIT = "c4412033401006b7e94b73cde3a05c61e045c12b"
ORIGEM = f"https://raw.githubusercontent.com/lidibelas/pesquisa_lol/{COMMIT}/"
PASTA_ENTRADA = os.environ.get("MOL_ENTRADA", "").strip()
PASTA_SAIDA = Path(os.environ.get("MOL_SAIDA", "resultados_mol"))
METADADOS = "data/data_base yiok - videos.csv"
LEXICOS = {
    "sem": ("lexico/mol-sem-frases-de-cena.csv", 90, 34),
    "com": ("lexico/mol-com-frases-de-cena.csv", 97, 40),
}
CATEGORIAS_AUX = ("racial", "homofobia_transfobia", "violencia_sexual")
MANIFESTO = {}
PALAVRAS = re.compile(r"[^\W_]+(?:['’\-][^\W_]+)*", re.UNICODE)

def ler_csv(caminho, campos):
    if PASTA_ENTRADA:
        dados = (Path(PASTA_ENTRADA) / caminho).read_bytes()
    else:
        pedido = Request(ORIGEM + quote(caminho, safe="/"), headers={"User-Agent": "Pesquisa-LoL-MOL/1"})
        with urlopen(pedido, timeout=35) as resposta:
            dados = resposta.read()
    MANIFESTO[caminho] = hashlib.sha256(dados).hexdigest()
    reader = csv.DictReader(io.StringIO(dados.decode("utf-8-sig"), newline=""))
    if not set(campos).issubset(reader.fieldnames or []):
        raise ValueError(f"Colunas inesperadas em {caminho}: {reader.fieldnames}")
    rows = list(reader)
    if any(None in row or any(row[k] is None for k in campos) for row in rows):
        raise ValueError(f"Linha CSV incompleta ou inválida em {caminho}.")
    return rows

def normalizar(s):
    return unicodedata.normalize("NFC", s).casefold()

def padrao_literal(s):
    termos = re.split(r"\s+", normalizar(s).strip())
    return re.compile(r"(?<!\w)" + r"\s+".join(map(re.escape, termos)) + r"(?!\w)")

def compilar(rows):
    itens = []
    vistos = set()
    for row in rows:
        chave = normalizar(row["termo"]).strip()
        if not chave or chave in vistos:
            raise ValueError(f"Entrada vazia ou duplicada: {row['termo']}")
        vistos.add(chave)
        status = row["status"]
        if status not in {"principal", "auxiliar", "contextual", "excluido"}:
            raise ValueError(f"Status inesperado: {status}")
        if status in {"contextual", "excluido"}:
            continue
        cat = row["categoria"]
        if status == "auxiliar" and cat not in CATEGORIAS_AUX:
            raise ValueError(f"Categoria auxiliar inesperada: {cat}")
        busca = padrao_literal(chave)
        excecoes = [padrao_literal(s) for s in row["excecoes_frase"].split("|") if s.strip()]
        itens.append((row["termo"], status, cat, busca, excecoes))
    return itens

def encontrar(texto, itens):
    # Cada candidato tem uma faixa de caracteres. A expressão mais longa
    # ganha de candidatos que ocupam os mesmos caracteres. A exceção só
    # bloqueia a faixa do termo em questão, não outras menções no segmento.
    texto = normalizar(texto)
    candidatos = []
    for termo, status, cat, busca, excecoes in itens:
        bloqueados = {
            (frase.start() + sub.start(), frase.start() + sub.end())
            for exc in excecoes
            for frase in exc.finditer(texto)
            for sub in busca.finditer(frase.group())
        }
        for match in busca.finditer(texto):
            if match.span() not in bloqueados:
                candidatos.append((match.start(), match.end(), termo, status, cat))
    aceitos = []
    for achado in sorted(candidatos, key=lambda x: (-(x[1] - x[0]), x[0], x[2])):
        inicio, fim = achado[:2]
        if not any(inicio < outro[1] and outro[0] < fim for outro in aceitos):
            aceitos.append(achado)
    return sorted(aceitos)

def medir(unidades, dic):
    n_palavras = sum(len(PALAVRAS.findall(normalizar(u))) for u in unidades)
    if not n_palavras:
        return None
    por_cat = Counter()
    flags_cat = Counter()
    termos = Counter()
    principal = flags_principal = 0
    for u in unidades:
        hits = encontrar(u, dic)
        n_principal = sum(h[3] == "principal" for h in hits)
        principal += n_principal
        flags_principal += bool(n_principal)
        por_cat.update(h[4] for h in hits)
        flags_cat.update({h[4] for h in hits})
        termos.update(h[2] for h in hits if h[3] == "principal")
    return {
        "n_palavras": n_palavras, "ocorrencias": principal, "flags": flags_principal,
        "taxa": principal * 1000 / n_palavras, "categorias": por_cat,
        "flags_categorias": flags_cat, "termos": termos,
    }

def ler_dicionarios():
    out = {}
    for versao, (p, total, n_principal) in LEXICOS.items():
        dados = ler_csv(p, ("termo", "categoria", "status", "excecoes_frase"))
        if len(dados) != total or sum(r["status"] == "principal" for r in dados) != n_principal:
            raise ValueError(f"Dicionário diferente do esperado: {p}")
        out[versao] = compilar(dados)
    return out

def testar(dics):
    for versao, dic in dics.items():
        def termos(txt):
            return [h[2] for h in encontrar(txt, dic)]
        assert not termos("Puta que pariu, filho da puta e filha da puta!")
        assert termos("Puta que pariu! A puta chegou. Putinha.") == ["puta", "putinha"]
        assert termos("Xputinha e boostadax não são palavras inteiras.") == []
        assert termos("web puta") == ["web puta"]
        assert termos("caralho, pica, foder, puto e putinho") == []
        assert termos("boostada e bustada") == ["boostada", "bustada"]
        assert termos("foi boostada pelo namorado") == (
            ["boostada"] if versao == "sem" else ["foi boostada pelo namorado"])
        assert termos("mulher não sabe jogar") == ([] if versao == "sem" else ["mulher não sabe jogar"])
        teste = medir(["estuprar um alvo", "putinha e putinha"], dic)
        assert (teste["ocorrencias"], teste["flags"], teste["flags_categorias"]["violencia_sexual"]) == (2, 1, 1)
        assert medir([""], dic) is None
        assert medir(["nenhuma palavra marcada"], dic)["taxa"] == 0
    print("Testes do léxico: OK (exceções, frases, ocorrências, flags e textos vazios).")

def resultados():
    dics = ler_dicionarios()
    testar(dics)
    campos = ("videoId", "videoTitle", "publishedAt", "viewCount", "tipo", "categoria_video", "videoDescription")
    metadados = ler_csv(METADADOS, campos)
    ids = [m["videoId"] for m in metadados]
    if len(ids) != 43 or len(set(ids)) != 43 or not all(re.fullmatch(r"[A-Za-z0-9_-]{11}", i) for i in ids):
        raise ValueError("Esperava 43 IDs de vídeo únicos e válidos.")
    if Counter(m["tipo"] for m in metadados) != {"longo": 22, "short": 21}:
        raise ValueError("Distribuição por formato inesperada.")
    categorias_princ = sorted({cat for dic in dics.values() for _,st,cat,_,_ in dic if st == "principal"})
    res = []
    status_descs = Counter()
    for m in metadados:
        vid = m["videoId"]
        segs = ler_csv(f"data/videos/{vid}/{vid}_transcricao.csv", ("texto", "segmento", "inicio_s", "fim_s"))
        if not segs or len({s["segmento"] for s in segs}) != len(segs):
            raise ValueError(f"Transcrição vazia ou com segmentos repetidos: {vid}")
        if any(s.get("video_id", vid) != vid for s in segs):
            raise ValueError(f"ID de transcrição não corresponde aos metadados: {vid}")
        audio = [s["texto"] for s in segs]
        # A descrição é uma única unidade. Excluímos URLs da descrição
        # para não contar componentes de links como se fossem palavras.
        descricao = re.sub(r"https?://\S+", " ", m["videoDescription"], flags=re.I)
        out = {
            "video_id": vid, "titulo": m["videoTitle"], "publicado_em": m["publishedAt"],
            "visualizacoes": int(m["viewCount"]), "formato": m["tipo"], "categoria_video": m["categoria_video"],
            "total_segmentos": len(audio),
            "status_descricao": "preenchida" if descricao.strip() else "vazia",
        }
        if out["visualizacoes"] < 0:
            raise ValueError(f"Contagem de views inválida: {vid}")
        status_descs[out["status_descricao"]] += 1
        for campo, unidades in (("fala", audio), ("descricao", [descricao])):
            for versao, dic in dics.items():
                med = medir(unidades, dic)
                if campo == "fala" and med is None:
                    raise ValueError(f"Transcrição sem palavras; não tratada como zero: {vid}")
                if versao == "sem":
                    out[f"{campo}_palavras"] = med["n_palavras"] if med else None
                out[f"{campo}_{versao}_ocorrencias"] = med["ocorrencias"] if med else None
                out[f"{campo}_{versao}_flags"] = med["flags"] if med else None
                out[f"{campo}_{versao}_por_mil"] = med["taxa"] if med else None
                for cat in CATEGORIAS_AUX:
                    out[f"{campo}_{versao}_{cat}_ocorrencias"] = med["categorias"][cat] if med else None
                    out[f"{campo}_{versao}_{cat}_flags"] = med["flags_categorias"][cat] if med else None
                if campo == "fala":
                    for cat in categorias_princ:
                        out[f"fala_{versao}_{cat}_ocorrencias"] = med["categorias"][cat] if med else None
                    out[f"fala_{versao}_termos"] = "; ".join(
                        f"{t}: {n}" for t, n in sorted(med["termos"].items())
                    )
        res.append(out)
    if status_descs != {"vazia": 14, "preenchida": 29}:
        raise ValueError(f"Distribuição de descrições inesperada: {status_descs}")
    if sum(r["total_segmentos"] for r in res) != 8384:
        raise ValueError("Número de segmentos inesperado.")
    print(f"Conferidos {len(res)} vídeos, 8.384 segmentos e {dict(status_descs)} descrições.")
    return res

def gravar_csv(rows):
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    destino = PASTA_SAIDA / "resultado-mol-yiok-43-videos.csv"
    with destino.open("w", encoding="utf-8-sig", newline="") as arquivo:
        writer = csv.DictWriter(arquivo, fieldnames=list(rows[0]))
        writer.writeheader()
        for row in rows:
            writer.writerow({k:(f"{v:.8f}" if isinstance(v, float) else v) for k,v in row.items()})
    print("CSV salvo em:", destino)
    return destino

def rankings(xs):
    pares = sorted(enumerate(xs), key=lambda p: p[1])
    ranks = [0] * len(xs)
    i = 0
    while i < len(pares):
        j = i + 1
        while j < len(pares) and pares[j][1] == pares[i][1]:
            j += 1
        medio = (i + 1 + j) / 2
        for orig, _ in pares[i:j]:
            ranks[orig] = medio
        i = j
    return ranks

def spearman(xs, ys):
    if len(xs) < 2 or len(set(xs)) < 2 or len(set(ys)) < 2:
        return None
    return statistics.correlation(rankings(xs), rankings(ys))

def resumir(rows):
    assert math.isclose(spearman([1, 2, 3], [10, 20, 30]), 1)
    assert math.isclose(spearman([1, 2, 3], [30, 20, 10]), -1)
    resumo = {}
    for grupo in ("todos", "longo", "short"):
        dados = rows if grupo == "todos" else [r for r in rows if r["formato"] == grupo]
        resumo[grupo] = {}
        for versao in ("sem", "com"):
            taxas = [r[f"fala_{versao}_por_mil"] for r in dados]
            rho = spearman(taxas, [r["visualizacoes"] for r in dados])
            resumo[grupo][versao] = dict(n=len(dados), rho=rho, mediana=statistics.median(taxas),
                ocorrencias=sum(r[f"fala_{versao}_ocorrencias"] for r in dados),
                flags=sum(r[f"fala_{versao}_flags"] for r in dados))
            print(grupo, versao, resumo[grupo][versao])
    mudou = [r["video_id"] for r in rows if r["fala_sem_ocorrencias"] != r["fala_com_ocorrencias"]]
    print("Vídeos com contagem principal diferente:", len(mudou), mudou)
    print("SHA-256 dos arquivos de entrada:")
    for nome, digest in sorted(MANIFESTO.items()):
        print(digest, nome)
    return resumo, mudou



In [ ]:
linhas = resultados()
arquivo_csv = gravar_csv(linhas)
resumo, videos_com_diferenca = resumir(linhas)


## Visualização
Cada ponto é um vídeo. Resultados para Shorts e longos são comparados separadamente na saída anterior.


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True, layout="constrained")
cores = {"longo": "#235b92", "short": "#ba591f"}
for ax, versao in zip(axs, ("sem", "com")):
    for tipo, nome, marcador in (("longo","Vídeos longos","o"), ("short","Shorts","^")):
        selecao = [r for r in linhas if r["formato"] == tipo]
        ax.scatter([r[f"fala_{versao}_por_mil"] for r in selecao],
                   [r["visualizacoes"] for r in selecao],
                   c=cores[tipo], marker=marcador, alpha=.75, edgecolors="white",
                   linewidths=.35, s=42, label=nome)
    rho = resumo["todos"][versao]["rho"]
    rho_txt = f"{rho:+.3f}" if rho is not None else "não definido"
    ax.set_title(f"{versao.capitalize()} frases de cena | ρ = {rho_txt}")
    ax.set_xlabel("Ocorrências por mil palavras transcritas")
    ax.grid(alpha=.22)
axs[0].set_ylabel("Visualizações na coleta")
axs[0].legend(frameon=False)
fig.suptitle("Canal Yiok: correspondências lexicais e visualizações")
fig.savefig(PASTA_SAIDA / "mol-yiok-views.png", dpi=160)
plt.show()



## Baixar o CSV (opcional)
Você pode baixar na aba de arquivos. Para baixar pelo caderno, mude a opção abaixo para True e rode esta célula.


In [ ]:
BAIXAR_CSV = False

if BAIXAR_CSV:
    from google.colab import files
    files.download(str(arquivo_csv))
